# Large-data model setup

Liesel-GAM can construct a model from a representative **basis setup sample** and encode the full data as a **raw observed position**. A batching or optimization library can then split that position without first materializing full basis matrices.

This notebook first demonstrates the stable Liesel-GAM preparation API. A final, clearly separated section shows the larger optimization workflow when Liesel's unmerged `optim-devel` functionality is available.

## Liesel-GAM: prepare the model and full-data position

The first part uses released Liesel APIs and runs independently of `liesel.optim`.

In [1]:
import liesel.model as lsl
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel_gam as gam

print("Liesel-GAM version:", gam.__version__)

Liesel-GAM version: 0.1.4


We use 1,000 observations, one P-spline covariate, and one random-intercept grouping variable. In an application, the full data can be much larger.

In [2]:
df = gam.demo_data(n=1_000, seed=1)
continuous = ["x_nonlin"]
categorical = ["x_cat"]

print("Full data shape:", df.shape)
print("Observed x_cat levels:", df["x_cat"].nunique())

Full data shape: (1000, 5)
Observed x_cat levels: 3


`category_coverage_indices()` returns positional rows that a splitter should retain in training. Here all rows are training rows so the stable preparation is self-contained.

In [3]:
required_train_indices = gam.category_coverage_indices(df, columns=categorical)
train_indices = list(range(len(df)))

print("Required category-coverage rows:", required_train_indices.tolist())
print("Eligible training rows:", len(train_indices))

Required category-coverage rows: [0, 4, 8]
Eligible training rows: 1000


`basis_setup_sample()` keeps the eligible continuous boundaries and categorical metadata, then fills the sample randomly without replacement.

In [4]:
setup_df = gam.basis_setup_sample(
    df,
    indices=train_indices,
    continuous=continuous,
    categorical=categorical,
    n=200,
    seed=2,
)

print("Setup data shape:", setup_df.shape)
setup_range = setup_df["x_nonlin"].agg(["min", "max"]).round(3).tolist()
print("Setup x_nonlin range:", setup_range)
print("Setup x_cat categories:", setup_df["x_cat"].cat.categories.tolist())

Setup data shape: (200, 5)
Setup x_nonlin range: [-1.992, 1.997]
Setup x_cat categories: ['a', 'b', 'c']


The representative data fixes the P-spline setup and the random-intercept category mapping.

In [5]:
def make_model(data):
    tb = gam.TermBuilder.from_df(data, approximation=True)
    loc = gam.AdditivePredictor(name="loc")
    loc += tb.ps("x_nonlin", k=20)
    loc += tb.ri("x_cat")
    y = lsl.Var.new_obs(
        data["y"].to_numpy(),
        distribution=lsl.Dist(tfd.Normal, loc=loc, scale=1.0),
        name="y",
    )
    return tb, lsl.Model([y])


tb, model = make_model(setup_df)
print("Observed model variables:", sorted(model.observed))

Observed model variables: ['x_cat', 'x_nonlin', 'y']


`PandasRegistry.observed_position()` now encodes exactly the observed model variables for all rows. It uses the category mapping established from `setup_df` and ignores unrelated DataFrame columns.

In [6]:
full_position = tb.registry.observed_position(model, df)
position_shapes = {
    name: tuple(value.shape) for name, value in sorted(full_position.items())
}

print("Full-position shapes:", position_shapes)
matches_observed = sorted(full_position) == sorted(model.observed)
print("Position matches model observations:", matches_observed)

Full-position shapes: {'x_cat': (1000,), 'x_nonlin': (1000,), 'y': (1000,)}
Position matches model observations: True


## Correctness constraints

Pass explicit column lists for a wide DataFrame. With `continuous=None` or `categorical=None`, columns are inferred by dtype; an empty list disables that column kind. Numeric category codes must first be cast to a pandas categorical dtype. Category levels outside the setup mapping are errors rather than silently receiving new codes.

Derived model quantities must be row-wise: one row's value may depend on that row and fixed setup metadata, but not on other rows in its current batch. Precompute lags, ranks, or other context-dependent features before batching. Setup-dependent knots, constraints, and penalty scaling are based on `setup_df`, not all training rows.

Do not call `consolidate_bases()` in this workflow. It intentionally turns derived bases into observed matrices and would materialize those matrices for the full data.

## Full picture with the unmerged Liesel optimization API

> **Warning**
>
> This section depends on `liesel.optim` from Liesel's unmerged `optim-devel` branch. It is not available in the released Liesel version or on Liesel's `main` branch, and its API may change before merging. The Liesel-GAM preparation above does not depend on it.

When available, Liesel owns splitting, validation, mini-batch construction, loss scaling, and optimization. Liesel-GAM only prepares representative setup data, constructs the GAM, and converts the full DataFrame into a compatible raw observed position.

In [7]:
try:
    import liesel.optim as opt  # ty: ignore[unresolved-import]
except ModuleNotFoundError as error:
    if error.name != "liesel.optim":
        raise
    opt = None
    print("Skipped: liesel.optim is not available in this Liesel installation.")
else:
    print("liesel.optim is available; running the short optimization example.")

Skipped: liesel.optim is not available in this Liesel installation.


In [8]:
if opt is None:
    print("Optimization skipped; the Liesel-GAM setup above completed successfully.")
else:
    n_validate = max(1, round(0.1 * len(df)))
    splitter = opt.Split(
        axis_size=len(df),
        validate_axis_size=n_validate,
        keep_in_train=required_train_indices,
        shuffle=True,
        seed=42,
    )
    optim_setup_df = gam.basis_setup_sample(
        df,
        indices=splitter.indices_train.tolist(),
        continuous=continuous,
        categorical=categorical,
        n=200,
        seed=43,
    )
    optim_tb, optim_model = make_model(optim_setup_df)
    optim_position = optim_tb.registry.observed_position(optim_model, df)
    split = splitter.split_position(optim_position)
    result = opt.LieselOptim(
        optim_model,
        split=split,
        batch_size=128,
        stopper=opt.Stopper(epochs=10, patience=5),
        seed=44,
        show_progress=False,
    ).fit()
    print("Split sizes:", split.train_axis_size, split.validate_axis_size)
    print("Optimization result:", type(result).__name__)

Optimization skipped; the Liesel-GAM setup above completed successfully.
